# Stable Diffusion Text-to-Image on Windows

这个 Notebook 用于在 Windows 环境中通过 Hugging Face 下载并运行官方 SDXL 模型 `stabilityai/stable-diffusion-xl-base-1.0`，完成基础文生图。

推荐环境：
- Windows 10/11
- Python 3.10 或 3.11
- NVIDIA GPU
- 建议 8GB 以上显存，12GB+ 体验更好

使用前请确认：
- 你已经在 Hugging Face 上接受 SDXL 模型协议
- 你可以登录 Hugging Face
- 你的 PyTorch 已经按本机 CUDA 版本正确安装

说明：这个 Notebook 主路径针对 GPU 设计。CPU 也能运行，但会明显更慢，不建议作为常用方案。

## 1. 安装依赖

先安装与本机 CUDA 匹配的 PyTorch。请不要盲目复制别人环境里的 `torch` 安装命令，建议直接去 PyTorch 官网选择你机器对应的安装方式：

- https://pytorch.org/get-started/locally/

安装好 PyTorch 之后，再运行下面这个单元安装其余依赖。

In [1]:
%pip install -U diffusers transformers accelerate safetensors huggingface_hub notebook ipywidgets


   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------------------------------- 0.0/645.5 kB ? eta -:--:--
   ---------------- ----------------------- 262.1/645.5 kB ? eta -:--:--
   -------------------------------- ------- 524.3/645.5 kB 1.3 MB/s eta 0:00:01
   ---------------------------------------- 645.5/645.5 kB 1.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.7 MB 2.1 MB/s eta 0:00:02
   ----------------- ---------------------- 1.6/3.7 MB 3.6 MB/s eta 0:00:01
   ------------------------------- -------- 2.9/3.7 MB 4.7 MB/s eta 0:00:01
   ------------------------------------- -- 3.4/3.7 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 3.5 MB/s  0:00:01
   ---------------------------------------- 0.0/10.2 MB ? eta -:--:--
   -------------------------------------

## 2. 检查 PyTorch 和 CUDA

这个单元会输出当前 PyTorch 版本、CUDA 是否可用，以及 GPU 名称。若 `CUDA available` 为 `False`，说明当前没有正确使用 NVIDIA GPU。

In [3]:
import platform
import sys

import torch

print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version reported by torch: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: CUDA is not available. The notebook can still run on CPU, but it will be much slower and is not the recommended path for SDXL.")

Python version: 3.10.20
Platform: Windows-10-10.0.26100-SP0
PyTorch version: 2.12.0.dev20260318+cu128
CUDA available: True
CUDA version reported by torch: 12.8
GPU count: 1
Current GPU: NVIDIA GeForce RTX 5070 Laptop GPU


## 3. 登录 Hugging Face

官方 SDXL 模型通常需要先在 Hugging Face 页面接受协议，再登录后才能下载。

模型页：
- https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0

下面提供两种方式：
- 交互式登录：适合 Notebook 环境
- 代码中直接传 token：适合你已经准备好 token 的情况

In [1]:
from huggingface_hub import login, notebook_login

# 方式 1：弹出交互式登录窗口
notebook_login()

# 方式 2：如果你已经有 token，也可以使用下面这行
# login(token="hf_xxx_your_token_here")

## 4. 配置模型与生成参数

这里把最常改的参数集中放在一个单元里，便于之后直接改 prompt 重复生成。

In [8]:
from pathlib import Path

MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

prompt = "A stunning panoramic view of Anfield stadium, home of Liverpool FC, vibrant match day atmosphere, packed stands with red-clad fans, lush green football pitch, bright stadium lights, clear blue sky, ultra-realistic, 8K, high detail, cinematic lighting, professional sports photography"
negative_prompt = "blurry, low quality, distorted, watermark, text"
num_inference_steps = 30
guidance_scale = 7.0
height = 1024
width = 1024
seed = 42

print(f"MODEL_ID: {MODEL_ID}")
print(f"DEVICE: {DEVICE}")
print(f"DTYPE: {DTYPE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")

MODEL_ID: stabilityai/stable-diffusion-xl-base-1.0
DEVICE: cuda
DTYPE: torch.float16
OUTPUT_DIR: D:\AI2026\AI Refactor\AI-refactor\DeepGen\sd\outputs


## 5. 下载并加载 SDXL Pipeline

首次运行时会从 Hugging Face 下载模型，速度取决于网络与模型缓存情况。

如果这里报错，常见原因包括：
- 还没有接受模型协议
- 还没有登录 Hugging Face
- 本地 CUDA / torch 环境不匹配
- 显存太小，无法顺利把模型移到 GPU

In [9]:
from diffusers import StableDiffusionXLPipeline

pipe = None

try:
    pipe = StableDiffusionXLPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=DTYPE,
        use_safetensors=True,
    )

    if hasattr(pipe, "enable_attention_slicing"):
        pipe.enable_attention_slicing()

    if hasattr(pipe, "enable_vae_slicing"):
        pipe.enable_vae_slicing()

    pipe = pipe.to(DEVICE)
    print("Pipeline loaded successfully.")
except Exception as exc:
    print("Failed to load the SDXL pipeline.")
    print(f"Error type: {type(exc).__name__}")
    print(f"Error message: {exc}")
    print()
    print("Checklist:")
    print("1. Make sure you accepted the SDXL model license on Hugging Face.")
    print("2. Make sure you have logged in successfully in the previous cell.")
    print("3. Make sure your torch installation matches your NVIDIA CUDA environment.")
    print("4. If you hit an out-of-memory error, lower the resolution or use a smaller model.")
    raise

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Pipeline loaded successfully.


## 6. 生成图片

这个单元会根据上面的 prompt 生成 1 张图片。设置固定 seed 后，可以得到可复现的结果。

In [10]:
from datetime import datetime

generator = torch.Generator(device=DEVICE)
generator.manual_seed(seed)

try:
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        height=height,
        width=width,
        generator=generator,
    )
    image = result.images[0]
    image
except torch.cuda.OutOfMemoryError:
    print("CUDA out of memory.")
    print("Try lowering height/width, reducing num_inference_steps, or switching to a lighter model such as SD 1.5.")
    raise
except Exception as exc:
    print("Image generation failed.")
    print(f"Error type: {type(exc).__name__}")
    print(f"Error message: {exc}")
    raise

  0%|          | 0/30 [00:00<?, ?it/s]

## 7. 保存图片

生成后的图片会保存到当前目录下的 `outputs` 文件夹。文件名会自动带上时间戳和 seed，避免覆盖。

In [11]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = OUTPUT_DIR / f"sdxl_{timestamp}_seed{seed}.png"
image.save(output_path)
print(f"Saved to: {output_path.resolve()}")

Saved to: D:\AI2026\AI Refactor\AI-refactor\DeepGen\sd\outputs\sdxl_20260422_193308_seed42.png


## 8. 常见问题排查

### 1. Hugging Face 下载失败
- 检查是否已经登录
- 检查是否已经在模型页面接受协议
- 检查网络是否能访问 Hugging Face

### 2. CUDA 不可用
- 检查是否安装了支持 CUDA 的 PyTorch，而不是 CPU 版
- 检查 NVIDIA 驱动是否正常
- 检查当前 Notebook 内核是否使用了正确的 Python 环境

### 3. 显存不足
- 把分辨率从 `1024 x 1024` 降到 `768 x 768`
- 减小 `num_inference_steps`
- 关闭其他占用显存的程序
- 如果显存较小，可以考虑换成 SD 1.5

### 4. 依赖安装问题
- 先确认 Python 版本是 3.10 或 3.11
- 优先在独立虚拟环境中安装
- 如果 `torch` 装错版本，先卸载后按 PyTorch 官网说明重新安装

### 5. 如何复现结果
- 固定相同的 `seed`
- 保持相同的 prompt、分辨率、步数和 guidance scale
- 尽量保持相同的软件版本